In [10]:
# Install dependencies jika belum terinstall
!pip install opencv-python mediapipe numpy tqdm



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
import urllib.request
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm

# 1. Download model Hand Landmarker jika belum ada di direktori
MODEL_PATH = 'hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Mengunduh model hand_landmarker.task...')
    url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('Model hand_landmarker.task berhasil diunduh!')

# 2. Inisialisasi MediaPipe Tasks API (modern API pengganti mp.solutions)
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

DEFAULT_DETECTOR = None

def get_detector():
    global DEFAULT_DETECTOR
    if DEFAULT_DETECTOR is None:
        base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
        options = vision.HandLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            num_hands=2,
            min_hand_detection_confidence=0.3,
            min_hand_presence_confidence=0.3
        )
        DEFAULT_DETECTOR = vision.HandLandmarker.create_from_options(options)
    return DEFAULT_DETECTOR

def extract_landmarks_from_video(video_path, sequence_length=30, detector=None):
    if detector is None:
        detector = get_detector()

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return None

    # Ambil 30 indeks frame secara merata sepanjang durasi video
    frame_indices = np.linspace(0, total_frames - 1, sequence_length, dtype=int)
    sequence_data = []

    current_frame_idx = 0
    target_ptr = 0

    while cap.isOpened() and target_ptr < sequence_length:
        ret, frame = cap.read()
        if not ret:
            break

        frame_features = None
        while target_ptr < sequence_length and current_frame_idx == frame_indices[target_ptr]:
            if frame_features is None:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                results = detector.detect(mp_image)

                left_hand = np.zeros(63, dtype=np.float32)
                right_hand = np.zeros(63, dtype=np.float32)

                if results.hand_landmarks and results.handedness:
                    for hand_landmarks, handedness_list in zip(results.hand_landmarks, results.handedness):
                        label = handedness_list[0].category_name or handedness_list[0].display_name

                        landmarks = []
                        for lm in hand_landmarks:
                            landmarks.extend([lm.x, lm.y, lm.z])
                        landmarks = np.array(landmarks, dtype=np.float32)

                        if label == 'Left':
                            left_hand = landmarks
                        elif label == 'Right':
                            right_hand = landmarks

                # 63 titik tangan kiri + 63 titik tangan kanan = 126 fitur per frame
                frame_features = np.concatenate([left_hand, right_hand])
            
            sequence_data.append(frame_features)
            target_ptr += 1

        current_frame_idx += 1

    cap.release()

    # Padding jika frame kurang dari sequence_length
    while len(sequence_data) < sequence_length:
        sequence_data.append(np.zeros(126, dtype=np.float32))

    return np.array(sequence_data, dtype=np.float32)

def process_dataset(base_dir):
    label_map = {}
    detector = get_detector()
    
    for split in ['Train', 'Testing']:
        split_path = os.path.join(base_dir, split)
        if not os.path.exists(split_path):
            print(f"Folder '{split}' tidak ditemukan di '{base_dir}', dilewati.")
            continue

        classes = sorted([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))])
        
        if not label_map:
            label_map = {cls_name: idx for idx, cls_name in enumerate(classes)}

        X_data = []
        y_data = []

        print(f"\n--- Memproses Folder: {split} ---")
        for cls_name in classes:
            cls_path = os.path.join(split_path, cls_name)
            label = label_map[cls_name]
            video_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]

            for vid_file in tqdm(video_files, desc=f"Kelas {cls_name}"):
                vid_path = os.path.join(cls_path, vid_file)
                features = extract_landmarks_from_video(vid_path, sequence_length=30, detector=detector)

                if features is not None:
                    X_data.append(features)
                    y_data.append(label)

        X_array = np.array(X_data, dtype=np.float32)
        y_array = np.array(y_data, dtype=np.int64)

        out_prefix = split.lower()
        np.save(f'X_{out_prefix}.npy', X_array)
        np.save(f'y_{out_prefix}.npy', y_array)
        
        print(f"Selesai {split}! Output X_{out_prefix}.npy shape: {X_array.shape}, y_{out_prefix}.npy shape: {y_array.shape}")

    with open('label_map.txt', 'w') as f:
        for cls_name, idx in label_map.items():
            f.write(f"{cls_name}:{idx}\n")
    print("\nPemetaan label berhasil disimpan di 'label_map.txt'")

if __name__ == '__main__':
    DATASET_DIR = './bisindo - video dataset'
    process_dataset(DATASET_DIR)



--- Memproses Folder: Train ---


Kelas Z: 100%|██████████| 2/2 [00:02<00:00,  1.07s/it]


Selesai Train! Output X_train.npy shape: (52, 30, 126), y_train.npy shape: (52,)

--- Memproses Folder: Testing ---


Kelas Z: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]

Selesai Testing! Output X_testing.npy shape: (26, 30, 126), y_testing.npy shape: (26,)

Pemetaan label berhasil disimpan di 'label_map.txt'


In [12]:
# Verifikasi hasil ekstraksi dataset
import numpy as np

X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_testing = np.load('X_testing.npy')
y_testing = np.load('y_testing.npy')

print(f"X_train shape   : {X_train.shape} (video, frame, koordinat)")
print(f"y_train shape   : {y_train.shape}")
print(f"X_testing shape : {X_testing.shape}")
print(f"y_testing shape : {y_testing.shape}")

with open('label_map.txt') as f:
    labels = [line.strip() for line in f.readlines()]
print(f"Total kelas terdeteksi: {len(labels)}")
print("Contoh pemetaan label:", labels[:5])


X_train shape   : (52, 30, 126) (video, frame, koordinat)
y_train shape   : (52,)
X_testing shape : (26, 30, 126)
y_testing shape : (26,)
Total kelas terdeteksi: 26
Contoh pemetaan label: ['A:0', 'B:1', 'C:2', 'D:3', 'E:4']


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load data hasil ekstraksi
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_testing = np.load('X_testing.npy')
y_testing = np.load('y_testing.npy')

# 2. Ubah label numerik ke One-Hot Encoding
num_classes = 26
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_testing_cat = to_categorical(y_testing, num_classes=num_classes)

# 3. Bangun Arsitektur Model LSTM
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(30, 126)),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 4. Latih Model
history = model.fit(
    X_train, y_train_cat,
    epochs=60,
    batch_size=8,
    validation_data=(X_testing, y_testing_cat),
    verbose=1
)

# 5. Evaluasi dan Cetak Hasil Metrik
test_loss, test_acc = model.evaluate(X_testing, y_testing_cat)
print(f"\nAkurasi Data Uji: {test_acc * 100:.2f}%")

y_pred = model.predict(X_testing)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\n--- Laporan Klasifikasi (Precision, Recall, F1-Score) ---")
print(classification_report(y_testing, y_pred_classes))

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        48,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         1,690 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,770 (342.85 KB)

 Trainable params: 87,770 (342.85 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 82ms/step - accuracy: 0.0192 - loss: 3.2773 - val_accuracy: 0.0000e+00 - val_loss: 3.2515
Epoch 2/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0192 - loss: 3.2279 - val_accuracy: 0.0385 - val_loss: 3.2211
Epoch 3/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0769 - loss: 3.1916 - val_accuracy: 0.0769 - val_loss: 3.1711
Epoch 4/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0577 - loss: 3.1317 - val_accuracy: 0.0769 - val_loss: 3.1091
Epoch 5/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0769 - loss: 3.0380 - val_accuracy: 0.0385 - val_loss: 3.0431
Epoch 6/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0769 - loss: 2.9345 - val_accuracy: 0.0769 - val_loss: 3.0140
Epoch 7/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.0769 - loss: 2.8883 - val_accuracy: 0.0769 - val_loss: 2.9615
Epoch 8/60
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0962 - loss: 2.8007 - val_accuracy: 0.0769 - val_loss: 2.9

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [15]:
# Simpan model setelah dievaluasi
model.save('model_lstm_bisindo.h5')
print("Model berhasil disimpan sebagai model_lstm_bisindo.h5")

Model berhasil disimpan sebagai model_lstm_bisindo.h5
